In [18]:
!pip install mlflow dagshub -q

import pandas as pd
import numpy as np
import mlflow
import mlflow.xgboost
import pickle

mlflow.set_tracking_uri("https://dagshub.com/mgior23/IEEE-Fraud-Detection.mlflow")
print("✅ Connected")

✅ Connected


In [19]:
model = mlflow.xgboost.load_model("models:/XGBoost_FraudDetection/1")
print("✅ Model loaded!")

✅ Model loaded!


In [20]:
client = mlflow.tracking.MlflowClient()
local_path = client.download_artifacts("86cc597066614a0c8bb1f82985918238", "xgboost_preprocessing.pkl", "/kaggle/working/")

with open(local_path, "rb") as f:
    prep_info = pickle.load(f)

print("✅ Preprocessing loaded")

✅ Preprocessing loaded


In [21]:
test_trans = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_id = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')

test = test_trans.merge(test_id, on='TransactionID', how='left')
test_ids = test['TransactionID']
X_test = test.drop(['TransactionID'], axis=1)

# Drop same columns as training
X_test = X_test.drop([c for c in prep_info['drop_cols'] if c in X_test.columns], axis=1)
X_test = X_test.drop([c for c in prep_info['high_card'] if c in X_test.columns], axis=1)

# Feature engineering
X_test['TransactionAmt_log'] = np.log1p(X_test['TransactionAmt'])
X_test['TransactionDT_day'] = X_test['TransactionDT'] // (24*3600)
X_test['TransactionDT_hour'] = (X_test['TransactionDT'] % (24*3600)) // 3600

for c in ['C1','C2','C3','C4']:
    if c in X_test.columns:
        X_test[f'{c}_to_amt'] = X_test[c] / (X_test['TransactionAmt'] + 1)

# Label encode
for col in prep_info['cat_cols']:
    if col in X_test.columns:
        X_test[col] = X_test[col].astype(str)
        le = prep_info['le_dict'][col]
        X_test[col] = X_test[col].map(
            lambda x, le=le: le.transform([x])[0] if x in le.classes_ else -1
        )

# Align to training columns
missing_cols = set(prep_info['feature_cols']) - set(X_test.columns)
for col in missing_cols:
    X_test[col] = 0

X_test = X_test[prep_info['feature_cols']]
print(f"✅ Test shape: {X_test.shape}")
print(f"Added {len(missing_cols)} missing columns as 0")

✅ Test shape: (506691, 424)
Added 26 missing columns as 0


In [22]:
preds = model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    'TransactionID': test_ids,
    'isFraud': preds
})

submission.to_csv('/kaggle/working/submission.csv', index=False)
print("✅ Submission saved!")
print(f"Fraud rate: {(preds > 0.5).mean():.4f}")
submission.head(10)

✅ Submission saved!
Fraud rate: 0.0202


,TransactionID,isFraud
0,3663549,0.012619
1,3663550,0.007550
2,3663551,0.019930
3,3663552,0.007151
4,3663553,0.009346
5,3663554,0.009241
6,3663555,0.065573
7,3663556,0.077469
8,3663557,0.001795
9,3663558,0.013562
